In [6]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from collections import Counter
from ai_captioning import image_captioning

file_path = 'dataset/NovaCore_Multimodal_Company_Report_2026.pdf'

In [7]:
def pdf_element_extracter(file_path):
    print(f"\n⏳ Initiating - pdf elements extraction...")
    elements = partition_pdf(
        filename=file_path, # local file path to the PDF you want to parse
        strategy="hi_res", # underlying method used to parse the document

        # detects table grids, parses row/column cells, and exports the data as clean HTML
        infer_table_structure=True,
        
        # Specifies which element types to extract images from (e.g., ["Image", "Table"]). Works best wth "hi_res"
        extract_image_block_types=["Image"],
        
        # convert images into base64 strings and embed them directly inside each element's metadata fields
        extract_image_block_to_payload=True,
        
        # # Groups narrative blocks logically by titles
        # chunking_strategy="by_title",
        
        # # Hard limit on number of characters to be part of a chunk
        # max_characters=2000
    )
    print(f"\n✅ Success!! Extracted {len(elements)} elements.")
    return elements

elements = pdf_element_extracter(file_path)


⏳ Initiating - pdf elements extraction...


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.



✅ Success!! Extracted 78 elements.


In [3]:
elements

In [4]:
elements[3].to_dict()

{'type': 'Table',
 'element_id': 'c14fbb424bfec5ad1cf1432683e5e942',
 'text': 'Headquarters Singapore Employees 1,480 2026 Revenue $132.0M Operating Margin 18.4% Primary Sector Industrial AI & IoT Markets Served 42 countries',
 'metadata': {'detection_class_prob': 0.8697316646575928,
  'is_extracted': 'partial',
  'coordinates': {'points': ((176.14950561523438, 1532.814208984375),
    (176.14950561523438, 1752.75341796875),
    (1478.72314453125, 1752.75341796875),
    (1478.72314453125, 1532.814208984375)),
   'system': 'PixelSpace',
   'layout_width': 1654,
   'layout_height': 2339},
  'last_modified': '2026-11-01T22:50:07',
  'text_as_html': '<table><tbody><tr><td>Headquarters</td><td>Singapore</td><td>Employees</td><td>1,480</td></tr><tr><td>2026 Revenue</td><td>$132.0M</td><td>Operating Margin</td><td>18.4%</td></tr><tr><td>Primary Sector</td><td>Industrial Al &amp; loT</td><td>Markets Served</td><td>42 countries</td></tr></tbody></table>',
  'filetype': 'application/pdf',
  'lang

In [8]:
# Groups atomic elements and chunks them to create Composite-Elements
def create_chunks_by_title(elements):
    chunks = chunk_by_title(
        elements, # The parsed atomic PDF elements comprising of titles, narrative-text, charts, tables etc
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2500, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks_by_title(elements)

✅ Created 11 chunks


In [9]:
chunks

In [10]:
def inspect_chunk_composition(chunks):
    print(f"\n🔍 Inspecting composition of {len(chunks)} chunks:\n")
    
    for i, chunk in enumerate(chunks):
        chunk_type = type(chunk).__name__
        
        # Check if the chunk is a CompositeElement (created by grouping text/elements)
        if hasattr(chunk.metadata, "orig_elements"):
            orig_elements = chunk.metadata.orig_elements
            # Extract the class names of all underlying atomic elements inside this chunk
            element_types = [type(el).__name__ for el in orig_elements]
        else:
            # Standalone elements like Table chunks might not have orig_elements
            element_types = [chunk_type]
            
        # Print summary for each chunk
        print(f"Chunk {i+1} [{chunk_type}]:")
        print(f"  - Character Length: {len(chunk.text)}")
        print(f"  - Constituent Element Types: {set(element_types)}") # Shows unique types inside
        print(f"  - Content Preview: {chunk.text[:120].strip()}...\n")

# Run inspection on your chunks
inspect_chunk_composition(chunks)


🔍 Inspecting composition of 11 chunks:

Chunk 1 [CompositeElement]:
  - Character Length: 780
  - Constituent Element Types: {'Title', 'Table', 'NarrativeText', 'Image'}
  - Content Preview: NOVACORE SYSTEMS LTD. FY2026 BUSINESS PERFORMANCE & OPERATIONS REPORT

A realistic fictional company document designed f...

Chunk 2 [CompositeElement]:
  - Character Length: 1204
  - Constituent Element Types: {'Title', 'NarrativeText'}
  - Content Preview: 1. Executive Overview

Narrative text section - useful for text retrieval and grounded summarization

NovaCore Systems L...

Chunk 3 [CompositeElement]:
  - Character Length: 612
  - Constituent Element Types: {'Title', 'NarrativeText', 'Table'}
  - Content Preview: 2026 Strategic Priorities

Priority 2026 target Year-end status European expansion >25% YoY regional revenue growth Achi...

Chunk 4 [CompositeElement]:
  - Character Length: 936
  - Constituent Element Types: {'NarrativeText', 'Table', 'Title', 'FigureCaption', 'Image'}
  - Conten

In [ ]:
for i, chunk in enumerate(chunks):
    # Check if the chunk has constituent elements
    if hasattr(chunk.metadata, "orig_elements"):
        for elem in chunk.metadata.orig_elements:
            if type(elem).__name__ == "Image":
                print(f"🎯 Found an image inside Chunk {i+1}!")
                
                # If you used extract_image_block_to_payload=True, the base64 string is right here:
                img_base64 = getattr(elem.metadata, "image_base64", None)
                if img_base64:
                    print(f"   - Base64 payload available! Length: {len(img_base64)} characters")
                    # TODO: Pass `img_base64` to Gemini 2.5 Flash for summarization!

🎯 Found an image inside Chunk 1!
   - Base64 payload available! Length: 82820 characters........
🎯 Found an image inside Chunk 4!
   - Base64 payload available! Length: 73176 characters........
🎯 Found an image inside Chunk 5!
   - Base64 payload available! Length: 72228 characters........
🎯 Found an image inside Chunk 6!
   - Base64 payload available! Length: 95348 characters........
🎯 Found an image inside Chunk 7!
   - Base64 payload available! Length: 81324 characters........
🎯 Found an image inside Chunk 7!
   - Base64 payload available! Length: 75768 characters........
🎯 Found an image inside Chunk 8!
   - Base64 payload available! Length: 66856 characters........
🎯 Found an image inside Chunk 9!
   - Base64 payload available! Length: 43264 characters........


In [13]:
chunks[1].to_dict()

{'type': 'CompositeElement',
 'element_id': '4b57ead5-16fe-4148-bf82-34d43313e2f8',
 'text': '1. Executive Overview\n\nNarrative text section - useful for text retrieval and grounded summarization\n\nNovaCore Systems Ltd. develops industrial AI gateways, computer-vision appliances, and connected sensor hubs for manufacturing, logistics, and energy companies. During FY2026, the company focused on three priorities: expanding enterprise deployments in Europe, improving the gross margin of its flagship AtlasEdge product line, and shortening post-deployment support response times.\n\nFY2026 revenue reached USD 132.0 million, up from USD 97.5 million in FY2025. Europe became the fastest-growing region, supported by automotive and warehouse automation projects. The AtlasEdge X1 remained the highest-margin product, while VisionHub Pro generated the largest number of new enterprise pilots. Management approved a further USD 11.5 million investment in the Penang assembly facility for test automat

In [14]:
chunks[1].metadata.orig_elements

In [15]:
chunks[0].text.split('\n\n')

['NOVACORE SYSTEMS LTD. FY2026 BUSINESS PERFORMANCE & OPERATIONS REPORT',
 'A realistic fictional company document designed for multimodal RAG testing',
 'Penang Smart Assembly Facili 2026 facility utilization: 82% Opened 2024 | 18,000 m2 | ISO 9001 certified Solar contribution to electricity: 34%',
 'Headquarters Singapore Employees 1,480 2026 Revenue $132.0M Operating Margin 18.4% Primary Sector Industrial AI & IoT Markets Served 42 countries',
 'Demo note: All company names, metrics, people, and events in this report are fictional. The document intentionally mixes text, tables, charts, and embedded images so it can be used to demonstrate multimodal retrieval and question answering.',
 'NovaCore Systems Ltd. - Fictional company report created for a multimodal RAG demonstration',
 'Page 1']

In [17]:
def extract_chunk_data_content(chunk):
    data_content = {
        'text':'',
        'image':[],
        'table':[],
        'data_types_counter':{}
    }

    data_types = []

    for atomic_element in chunk.metadata.orig_elements:
        # element_type can be Image, Table, Title, NarrativeText, ListItem
        element_type = str(type(atomic_element).__name__)

        if element_type == 'Image':
            data_content['image'].append(atomic_element.metadata.image_base64)
            data_types.append('image')

        elif element_type == 'Table':
            data_content['table'].append(atomic_element.metadata.text_as_html)
            data_types.append('table')
        
        else:
            data_content['text'] = data_content['text'] + ' \n\n '+ atomic_element.text
            data_types.append('text')
    
    # Count the frequency of each datatype and store in dictionary
    data_content['data_types_counter'] = dict(Counter(data_types))
    return(data_content)

def generate_textual_summary(data_chunk):
    if data_chunk['image']:
        for img_index, img in enumerate(data_chunk['image']):
            img_summary = image_captioning(img)
            print(img_summary)
    
    # if data_chunk['table']:
    #     for table_index, table in enumerate(data_chunk['table']):
    #         table_summary = table_captioning(table) # table is stored in html format

for chunk_index, chunk in enumerate(chunks):
    chunk_number = chunk_index + 1
    chunk_data_content = extract_chunk_data_content(chunk)
    print(f"------ Processed chunk {chunk_number} ------ {chunk_data_content['data_types_counter']}")
    generate_textual_summary(chunk_data_content)
    break

------ Processed chunk 1 ------ {'text': 5, 'image': 1, 'table': 1}
[{'type': 'text', 'text': "**Title & Context:** The graphic illustrates an overview of the Penang Smart Assembly Facility, detailing its foundational specifications, operational metrics, and sustainability integration. It addresses key facility-level performance indicators, specifically answering questions regarding its physical dimensions, operational status, and renewable energy contribution.\n\n**Visual Components:** The infographic is presented as a digital illustration of an industrial warehouse building, accompanied by two text callout boxes. The top-left callout outlines facility metadata, while the top-right callout displays performance metrics. Visual elements on the building include solar panels mounted on the roof, evenly spaced windows, a loading bay door, and a delivery truck parked outside next to trees.\n\n**Key Data Points & Trends:** The facility officially opened in 2024, spans a total area of 18,000 